<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/intermediate/01_ml_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Fundamentals — Bias, Variance & Cross-Validation

Understanding these concepts separates data scientists who build models from those who build *good* models.

**Topics:** Bias-variance tradeoff, overfitting/underfitting, cross-validation, regularization, learning curves

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import (cross_val_score, KFold, StratifiedKFold, 
                                      learning_curve, validation_curve)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification, make_regression
from sklearn.metrics import mean_squared_error

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Bias-Variance Tradeoff — Visual Demonstration

In [ ]:
# True function: f(x) = sin(x)
# We observe noisy samples: y = f(x) + ε

def true_fn(x): return np.sin(x)

def generate_data(n, noise=0.3):
    x = np.sort(np.random.uniform(0, 2*np.pi, n))
    y = true_fn(x) + np.random.randn(n) * noise
    return x, y

x_test = np.linspace(0, 2*np.pi, 200)
y_true = true_fn(x_test)

# Compare models: underfitting (deg=1), good (deg=4), overfitting (deg=15)
n_experiments = 20
degrees = [1, 4, 15]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, deg in zip(axes, degrees):
    predictions = []
    for _ in range(n_experiments):
        x_train, y_train = generate_data(20)
        poly = PolynomialFeatures(deg, include_bias=False)
        X_train = poly.fit_transform(x_train.reshape(-1,1))
        X_test_poly = poly.transform(x_test.reshape(-1,1))
        model = LinearRegression()
        model.fit(X_train, y_train)
        predictions.append(model.predict(X_test_poly))
    
    preds = np.array(predictions)
    mean_pred = preds.mean(axis=0)
    bias_sq = (mean_pred - y_true)**2
    variance = preds.var(axis=0)
    
    # Plot a few individual fits
    for pred in predictions[:5]:
        ax.plot(x_test, pred, 'b-', alpha=0.2, lw=1)
    ax.plot(x_test, mean_pred, 'b-', lw=2, label='Mean prediction')
    ax.plot(x_test, y_true, 'r-', lw=2, label='True function')
    
    ax.set_title(f'Degree {deg}\nBias²={bias_sq.mean():.3f} | Var={variance.mean():.3f}')
    ax.set_ylim(-2.5, 2.5); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

axes[0].set_title(f'Degree 1 (UNDERFITTING)\n← High Bias, Low Variance')
axes[1].set_title(f'Degree 4 (JUST RIGHT)\n← Good Balance')
axes[2].set_title(f'Degree 15 (OVERFITTING)\n← Low Bias, High Variance')

plt.suptitle('Bias-Variance Tradeoff: 20 Models Trained on Different Samples', fontsize=13)
plt.tight_layout(); plt.show()

## 2. Cross-Validation

In [ ]:
X, y = make_classification(n_samples=500, n_features=20, n_informative=10, random_state=42)
clf = DecisionTreeClassifier(max_depth=5, random_state=42)

# 5-fold cross-validation
cv_scores_5 = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
cv_scores_10 = cross_val_score(clf, X, y, cv=10, scoring='accuracy')
cv_stratified = cross_val_score(clf, X, y, cv=StratifiedKFold(5), scoring='accuracy')

print('Cross-Validation Results:')
print(f'  5-Fold CV:     {cv_scores_5.mean():.4f} ± {cv_scores_5.std():.4f}  scores={cv_scores_5.round(3)}')
print(f'  10-Fold CV:    {cv_scores_10.mean():.4f} ± {cv_scores_10.std():.4f}')
print(f'  Stratified-5:  {cv_stratified.mean():.4f} ± {cv_stratified.std():.4f}')

# Show the K-fold splits visually
fig, ax = plt.subplots(figsize=(10, 4))
kf = KFold(n_splits=5, shuffle=True, random_state=42)
n = len(y)
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X)):
    ax.scatter(train_idx, [fold_idx]*len(train_idx), c='steelblue', s=1, alpha=0.5)
    ax.scatter(val_idx, [fold_idx]*len(val_idx), c='red', s=1)
ax.set_yticks(range(5)); ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_xlabel('Sample index')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='steelblue', label='Training'), Patch(color='red', label='Validation')])
ax.set_title('5-Fold Cross-Validation Splits')
plt.tight_layout(); plt.show()

## 3. Regularization — L1 (Lasso) vs L2 (Ridge)

In [ ]:
# Regularization adds a penalty to prevent large coefficients
# L2 (Ridge): ||β||² — shrinks all coefficients
# L1 (Lasso): ||β||₁ — sets some coefficients to exactly 0 (feature selection!)

X_reg, y_reg, true_coef = make_regression(n_samples=100, n_features=20, n_informative=5, 
                                            coef=True, random_state=42)

alphas = np.logspace(-3, 3, 50)
ridge_coefs = [Ridge(alpha=a).fit(X_reg, y_reg).coef_ for a in alphas]
lasso_coefs = [Lasso(alpha=a, max_iter=5000).fit(X_reg, y_reg).coef_ for a in alphas]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge: coefficients approach 0 but never reach it
ax = axes[0]
for coef_path in np.array(ridge_coefs).T:
    ax.semilogx(alphas, coef_path, lw=1, alpha=0.7)
ax.axvline(1.0, color='black', ls='--', label='α=1')
ax.set_xlabel('Regularization strength α'); ax.set_ylabel('Coefficient value')
ax.set_title('Ridge (L2): All coefficients shrink\nbut never reach exactly 0')
ax.legend(); ax.grid(True, alpha=0.3)

# Lasso: many coefficients go to exactly 0
ax2 = axes[1]
for coef_path in np.array(lasso_coefs).T:
    ax2.semilogx(alphas, coef_path, lw=1, alpha=0.7)
ax2.axvline(1.0, color='black', ls='--', label='α=1')
ax2.set_xlabel('Regularization strength α'); ax2.set_ylabel('Coefficient value')
ax2.set_title('Lasso (L1): Coefficients go to exactly 0\n→ automatic feature selection!')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# Non-zero coefficients at various alpha values
print('Lasso non-zero features at different α:')
for a in [0.01, 0.1, 1.0, 10.0]:
    lasso = Lasso(alpha=a, max_iter=5000).fit(X_reg, y_reg)
    n_nonzero = (lasso.coef_ != 0).sum()
    print(f'  α={a:5}: {n_nonzero}/20 non-zero features')

## 4. Learning Curves — Diagnosing Model Problems

In [ ]:
# Learning curves show train vs validation score as training size increases
# Diagnoses: underfitting (both scores low), overfitting (gap between train & val)

X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (model, title) in zip(axes, [
    (DecisionTreeClassifier(max_depth=2), 'Underfitting (max_depth=2)'),
    (DecisionTreeClassifier(max_depth=None), 'Overfitting (max_depth=None)'),
]):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
    )
    
    ax.plot(train_sizes, train_scores.mean(axis=1), 'b-o', lw=2, label='Train score')
    ax.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                    train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.2, color='blue')
    ax.plot(train_sizes, val_scores.mean(axis=1), 'r-o', lw=2, label='Validation score')
    ax.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                    val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.2, color='red')
    ax.set_xlabel('Training samples'); ax.set_ylabel('Accuracy')
    ax.set_title(title); ax.legend(); ax.set_ylim(0.5, 1.05)
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves: Diagnosing Bias vs Variance', fontsize=13)
plt.tight_layout(); plt.show()

print('Interpretation:')
print('  High bias:     Both curves are low and converged (add complexity)')
print('  High variance: Large gap between train and val curves (regularize, add data)')
print('  Just right:    Both curves are high and close together')